# Session 05: Practice Quiz

**Notebook 2 of 2: Self-check** · about 35 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one. The code cells use `httpx.MockTransport` or respx, so no network is needed.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

In [ ]:
import httpx
import respx

print("httpx", httpx.__version__, "| respx", respx.__version__)

## Q1 · Predict the output

```python
def server(request):
    return httpx.Response(503, json={"error": "maintenance"})

client = httpx.Client(transport=httpx.MockTransport(server))
response = client.get("http://inventory.test/assets/db-03")
print(response.status_code, response.is_success, response.json())
```

Is an exception raised? Run it to check.

In [ ]:
def server(request):
    return httpx.Response(503, json={"error": "maintenance"})


with httpx.Client(transport=httpx.MockTransport(server)) as client:
    response = client.get("http://inventory.test/assets/db-03")
    print(response.status_code, response.is_success, response.json())

<details>
<summary>Show answer</summary>

`503 False {'error': 'maintenance'}`, and **no exception**.

An error status is a perfectly valid HTTP response. HTTPX raises only when it gets **no** response (timeouts, connection failures) or when you ask it to, with `raise_for_status()`. Code that does `response.json()["name"]` without checking the status will fail later, with a confusing `KeyError` instead of a clear "service unavailable".

</details>

## Q2 · Which exception?

For each situation, name the HTTPX exception (if any) that `client.get(...)` raises, without `raise_for_status()`:

1. The server is up but takes 30 seconds; your timeout is 5.
2. Nothing listens on the port.
3. The server answers `404`.
4. The DNS name doesn't exist.
5. The server answers `200` with an HTML maintenance page, and you call `response.json()`.

<details>
<summary>Show answer</summary>

1. `httpx.ReadTimeout` (a `TimeoutException`, a `TransportError`).
2. `httpx.ConnectError` ("connection refused"). It's instant on Linux and macOS; Windows retries for a second or two, so a very short timeout can turn it into `ConnectTimeout`.
3. None. You get a `Response` with `status_code == 404`.
4. `httpx.ConnectError`. The name lookup happens as part of connecting.
5. Not an HTTPX error: `json.JSONDecodeError` from `.json()` (a `ValueError`). Your client should translate that into its own error, too.

Catching `httpx.TransportError` covers 1, 2 and 4. Status codes (3) and body parsing (5) are your code's responsibility.

</details>

## Q3 · Debugging: the nightly job that never finished

```python
import requests

def enrich(tickets):
    for ticket in tickets:
        response = requests.get(f"{INVENTORY_URL}/assets/{ticket['asset_id']}",
                                headers={"Authorization": f"Bearer {TOKEN}"})
        ticket["asset"] = response.json()
    return tickets
```

On the night of an inventory deployment, the job was still running in the morning. Find **four** problems.

<details>
<summary>Show answer</summary>

1. **No timeout.** `requests` waits forever by default, so one stuck connection hangs the job. Always pass `timeout=`, or use HTTPX, which defaults to 5 seconds.
2. **No status check.** A `503` or `401` body is stored as if it were an asset. Check `status_code` before using the body.
3. **No error handling.** A `ConnectionError` crashes the whole job. Decide what an unavailable inventory means for the report (probably "report without enrichment, and say so") and handle it there.
4. **A new connection per request.** Module-level `requests.get` creates a new session each time. Use one client (`requests.Session` or `httpx.Client`) for the whole loop.

Bonus: it **mutates its input** (`ticket["asset"] = ...`, Session 02), and it reads `TOKEN` from a module global; check that it comes from the environment, not the source.

</details>

## Q4 · Engineering judgment: what should the client do?

Your `InventoryClient.get_asset()` receives each of these. What should it do: return a value, raise which kind of error, or retry?

1. `404 {"error": "asset 'db-99' not found"}`
2. `401 {"error": "missing or invalid token"}`
3. `503` with `Retry-After: 2`
4. `200` with a body missing the `"team"` field
5. A `ReadTimeout` on `POST /assets/db-03/notes`

<details>
<summary>Show answer</summary>

1. Raise `AssetNotFound`, a normal, expected answer that callers can handle (or return `None`, if that's your client's documented contract). Don't retry.
2. Raise an auth error. It's a configuration problem; retrying won't help. The message must not include the token.
3. Raise "unavailable", or retry a small number of times with backoff, since `GET` is idempotent. Honour `Retry-After` if you retry.
4. Raise `InventoryError`, an unexpected answer. Don't silently return a half-filled object. (Session 06's Pydantic models make this check automatic.)
5. Raise "unavailable", and **don't retry automatically**: the server may already have created the note, and a retry would duplicate it. Let the caller decide, or use an idempotency key if the API supports one.

</details>

## Q5 · Predict the output: `.env` and the environment

In [ ]:
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv

os.environ["REPORT_TIMEOUT"] = "10"

with tempfile.TemporaryDirectory() as workdir:
    env_file = Path(workdir) / ".env"
    env_file.write_text("REPORT_TIMEOUT=30\nREPORT_OWNER=platform\n", encoding="utf-8")
    load_dotenv(env_file)

timeout = os.environ["REPORT_TIMEOUT"]
print(timeout, os.environ["REPORT_OWNER"], type(timeout).__name__)

for name in ["REPORT_TIMEOUT", "REPORT_OWNER"]:
    os.environ.pop(name, None)

<details>
<summary>Show answer</summary>

`10 platform str`

- `REPORT_TIMEOUT` was already set, so the `.env` value doesn't override it. In CI and production, real environment variables must win.
- `REPORT_OWNER` wasn't set, so it comes from the file.
- Environment variables are **always strings**. `"10"` must be converted (`float(...)`) and validated, because `"ten"` or `"0"` are just as possible. Session 06's `pydantic-settings` automates this.

</details>

## Q6 · Debugging: a leaked token

A support ticket includes this log line from the asset-lookup tool:

```text
ERROR inventory refused request: Settings(base_url='https://inventory.internal', token='inv_live_8f2a91c4d7', timeout_seconds=5.0)
```

1. What went wrong, and what must happen **right now**?
2. Name two code changes that stop it happening again.

<details>
<summary>Show answer</summary>

1. The settings object was logged, and its `repr` includes the token. **Revoke the token immediately** and issue a new one: it's now in the log system, the support ticket, and possibly email. Deleting the log line doesn't undo a leak.
2. Any two of:
   - `token: str = field(repr=False)` on the dataclass, so `repr()` and f-strings with `!r` never show it;
   - log *which* request failed and its status, never the settings or headers;
   - have error messages name the variable (`INVENTORY_API_TOKEN`), never its value;
   - add a test asserting that the token string doesn't appear in error messages or `repr()`, as the workbook's acceptance tests do.

</details>

## Q7 · Implementation: test a 404 with respx

`find_owner(client, asset_id)` should return the owning team, or `None` if the asset doesn't exist. Complete the respx route definitions so that both assertions pass.

In [ ]:
def find_owner(client: httpx.Client, asset_id: str) -> str | None:
    response = client.get(f"/assets/{asset_id}")
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()["team"]


with (
    respx.mock(base_url="http://inventory.test") as router,
    httpx.Client(base_url="http://inventory.test") as client,
):
    # Define two routes on `router` here: db-03 returns {"team": "database"}, nope returns 404.

    try:
        assert find_owner(client, "db-03") == "database"
        assert find_owner(client, "nope") is None
        print("both assertions passed")
    except Exception as error:
        print(f"{type(error).__name__}: {error}")

<details>
<summary>Show answer</summary>

```python
    router.get("/assets/db-03").respond(200, json={"team": "database"})
    router.get("/assets/nope").respond(404, json={"error": "asset 'nope' not found"})
```

(Indented four spaces, inside the `with` block, where the comment is.)

Before you add the routes, the cell prints an error saying that the request was **not mocked**. Inside `respx.mock`, an unexpected request fails loudly instead of reaching the network, which is exactly what you want in tests.

</details>

## Q8 · Predict: does the timeout fire?

In [ ]:
def slow_server(request):
    raise httpx.ReadTimeout("simulated: no answer within the read timeout", request=request)


with httpx.Client(transport=httpx.MockTransport(slow_server), timeout=1.0) as client:
    try:
        client.get("http://inventory.test/assets/slow-01")
    except httpx.TransportError as error:
        is_timeout = isinstance(error, httpx.TimeoutException)
        print("caught", type(error).__name__, "| is a TimeoutException:", is_timeout)

<details>
<summary>Show answer</summary>

`caught ReadTimeout | is a TimeoutException: True`

Two lessons:

- `except httpx.TransportError` catches timeouts too, because `TimeoutException` is a subclass of `TransportError`. Put a more specific `except httpx.TimeoutException` **first** if you want a different message for timeouts.
- A mock transport doesn't actually wait. It simulates the *outcome* of a timeout instantly, which keeps tests fast. Testing the real timing needs a real server, as in the workbook's integration tests.

</details>

## Q9 · Comparison

1. In C#, `HttpClient` instances are long-lived and usually created through `IHttpClientFactory`. What's the HTTPX equivalent practice, and why do both ecosystems care?
2. Java's `HttpClient.send` and C#'s `GetAsync` don't throw on `404`. JavaScript's `fetch` doesn't either. What's the shared design idea, and how does HTTPX follow it?

<details>
<summary>Show answer</summary>

1. Create **one `httpx.Client`** per service (or per application), reuse it, and close it on shutdown, for example by wrapping it in your own client class with a `with` block. Both ecosystems care for the same reason: a client owns a **connection pool**. Creating one per request throws the pool away, paying connection setup every time and, under load, exhausting sockets (the famous C# `HttpClient` socket-exhaustion problem).
2. A status code is a **valid HTTP outcome**, not a transport failure, so it comes back as data. Exceptions are reserved for "there's no response at all". HTTPX does the same: `RequestError` subclasses for transport failures, normal `Response` objects for any status, and an opt-in `raise_for_status()`, like C#'s `EnsureSuccessStatusCode()`.

</details>

## Q10 · Engineering judgment: where does the token go?

Rank these places to keep the inventory token for a developer's local runs, from best to worst, and say what's wrong with the bad ones:

1. Hard-coded in `settings.py`, "just for now".
2. In `.env`, which is listed in `.gitignore`, with a placeholder in `.env.example`.
3. In `.env.example`, "so the team has it".
4. Exported in the shell profile (`~/.bashrc`, or the user environment on Windows).
5. Passed as a command-line argument: `asset-lookup --token inv_live_... show db-03`.

<details>
<summary>Show answer</summary>

Best to worst: **2, 4, 5, 3, 1.**

- **2** is the standard: per project, never committed, with documented variables.
- **4** works, but the token is now visible to *every* program and project you run, and easy to forget about.
- **5** leaks into shell history, and into process listings that other users on the machine can see.
- **3** commits the real token. `.env.example` is meant for placeholders.
- **1** commits the token **and** couples the code to one environment. "Just for now" lasts until the repository is shared.

In CI and production, none of these apply. The platform injects the variable from a secrets store, and the same `load_settings()` code reads it.

</details>

## How did you do?

- **Q1, Q2, Q8** cover responses versus exceptions: Parts 1 and 3 of `01-concepts.ipynb`.
- **Q3, Q4** cover resilient client design: Parts 3, 5 and 7.
- **Q5, Q6, Q10** cover configuration and secrets: Part 4.
- **Q7** covers testing with respx: Part 6.
- **Q9** compares HTTPX with Java, C# and JavaScript.

Next: the workbook in [`../workbook/assignment.md`](../workbook/assignment.md).